# Colab Frontend + Backend App Test

This notebook starts the real FastAPI backend, serves the real frontend, checks streaming directly, and then opens the app through ngrok.


In [ ]:
# Cell 1: mount Drive and install runtime packages

from google.colab import drive

drive.mount('/content/drive')

!pip -q install fastapi uvicorn pyngrok tavily-python google-genai transformers sentence-transformers torch pandas requests


In [ ]:
# Cell 2: set project paths

import os
import sys

project_root = os.path.join(
    '/content',
    'drive',
    'MyDrive',
    'UoP',
    'COMP3000',
    'dual_dimension_misinformation_analyzer',
)
backend_root = os.path.join(project_root, 'backend')
frontend_root = os.path.join(project_root, 'frontend')
test_root = os.path.join(project_root, 'test')

if backend_root not in sys.path:
    sys.path.insert(0, backend_root)

print('project_root:', project_root)
print('backend_root:', backend_root)
print('frontend_root:', frontend_root)
print('test_root:', test_root)


In [ ]:
# Cell 3: set API keys

import os

# Keep your existing plaintext key lines here if you prefer.
# os.environ['TAVILY_API_KEY'] = '...'
# os.environ['GEMINI_API_KEY'] = '...'
# os.environ['NGROK_AUTHTOKEN'] = '...'

try:
    from google.colab import userdata

    for key_name in ['TAVILY_API_KEY', 'GEMINI_API_KEY', 'NGROK_AUTHTOKEN']:
        if not os.environ.get(key_name):
            secret_value = userdata.get(key_name)
            if secret_value:
                os.environ[key_name] = secret_value
except Exception:
    pass

print('TAVILY_API_KEY:', 'set' if os.environ.get('TAVILY_API_KEY') else 'missing')
print('GEMINI_API_KEY:', 'set' if os.environ.get('GEMINI_API_KEY') else 'missing')
print('NGROK_AUTHTOKEN:', 'set' if os.environ.get('NGROK_AUTHTOKEN') else 'missing')


In [ ]:
# Cell 4: check files and backend imports

import os

required_files = [
    os.path.join(backend_root, 'app.py'),
    os.path.join(backend_root, 'analysis_orchestrator.py'),
    os.path.join(backend_root, 'fact_checking', 'fact_check_service.py'),
    os.path.join(backend_root, 'fact_checking', 'recovery.py'),
    os.path.join(frontend_root, 'index.html'),
    os.path.join(frontend_root, 'script.js'),
    os.path.join(frontend_root, 'style.css'),
]

missing_files = [file_path for file_path in required_files if not os.path.exists(file_path)]
if missing_files:
    raise FileNotFoundError('
'.join(missing_files))

os.chdir(backend_root)

from fact_checking.gemini_agent import is_gemini_available

print('Required files OK.')
print('Gemini available:', is_gemini_available())


In [ ]:
# Cell 5: stop old app server and ngrok tunnels

import os
import signal
import subprocess

from pyngrok import ngrok

if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=10)

if 'server_log_file' in globals() and not server_log_file.closed:
    server_log_file.close()

ngrok.kill()

print('Old server and ngrok tunnels stopped.')


In [ ]:
# Cell 6: start FastAPI app

import os
import subprocess
import time
import requests

os.chdir(backend_root)

server_log_path = os.path.join(test_root, 'frontend_server.log')
server_log_file = open(server_log_path, 'w', encoding='utf-8')

server_process = subprocess.Popen(
    [
        'python',
        '-u',
        '-m',
        'uvicorn',
        'app:app',
        '--host',
        '127.0.0.1',
        '--port',
        '8000',
        '--log-level',
        'warning',
    ],
    stdout=server_log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

for attempt_index in range(45):
    if server_process.poll() is not None:
        server_log_file.flush()
        with open(server_log_path, 'r', encoding='utf-8', errors='replace') as log_file:
            print(log_file.read()[-4000:])
        raise RuntimeError('FastAPI failed to start.')

    try:
        response = requests.get('http://127.0.0.1:8000/script.js', timeout=5)
        if response.status_code == 200:
            print('FastAPI is running on port 8000.')
            print('server log:', server_log_path)
            break
    except Exception:
        pass

    time.sleep(2)
else:
    print('FastAPI is still not reachable.')
    print('server log:', server_log_path)


In [ ]:
# Cell 7: quick local app check

import requests

home_response = requests.get('http://127.0.0.1:8000/', timeout=20)
script_response = requests.get('http://127.0.0.1:8000/script.js', timeout=20)
api_response = requests.post(
    'http://127.0.0.1:8000/analyze',
    json={'claim': '', 'options': {'use_query_rewrite': True}},
    timeout=20,
)

print('home:', home_response.status_code)
print('script.js:', script_response.status_code)
print('analyze invalid-input check:', api_response.status_code)
print(api_response.text[:500])


In [ ]:
# Cell 8: test streaming locally without browser

import json
import time
import requests

payload = {
    'claim': 'The Earth orbits the Sun.',
    'options': {
        'use_query_rewrite': True,
        'relevance_threshold': 0.35,
        'use_oversampling_retry': True,
        'use_selective_stabilization': True,
        'top_k': 3,
        'use_all_eligible_evidence': False,
        'retrieval_results': 8,
    },
}

start_time = time.time()

with requests.post(
    'http://127.0.0.1:8000/analyze/stream',
    json=payload,
    stream=True,
    timeout=(10, 600),
) as response:
    print('status:', response.status_code)

    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue

        event = json.loads(line.removeprefix('data: '))
        elapsed = round(time.time() - start_time, 1)

        if event['event'] == 'progress':
            data = event['data']
            print(elapsed, 'progress:', data.get('stage'), '-', data.get('message'))
        elif event['event'] == 'result':
            data = event['data']
            print(elapsed, 'result:', data.get('status'))
            print('fact status:', data.get('fact_checking', {}).get('status'))
            break
        elif event['event'] == 'error':
            print(elapsed, 'error:', event['data'])
            break


In [ ]:
# Cell 9: open public ngrok URL

import os
from pyngrok import ngrok

ngrok_token = os.environ.get('NGROK_AUTHTOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(addr='127.0.0.1:8000', proto='http').public_url

print('Public app URL:')
print(public_url)


In [ ]:
# Cell 10: quick public app check

import time
import requests

home_response = None
script_response = None

for attempt_index in range(30):
    home_response = requests.get(public_url, timeout=10)
    script_response = requests.get(public_url + '/script.js', timeout=10)

    print(
        f'attempt {attempt_index + 1}/30 '
        f'home={home_response.status_code} script={script_response.status_code}'
    )

    if home_response.status_code == 200 and script_response.status_code == 200:
        break

    time.sleep(2)

if home_response.status_code != 200 or script_response.status_code != 200:
    print('home preview:')
    print(home_response.text[:500])
    print('script.js preview:')
    print(script_response.text[:500])
    raise RuntimeError('The public app URL is not serving the frontend correctly yet.')

api_response = requests.post(
    public_url + '/analyze',
    json={'claim': '', 'options': {'use_query_rewrite': True}},
    timeout=20,
)

print('analyze invalid-input check:', api_response.status_code)
print(api_response.text[:500])


In [ ]:
# Cell 11: check served frontend script

import time
import requests

script_url = public_url + f'/script.js?check={int(time.time())}'
script_response = requests.get(script_url, timeout=20)

print('script status:', script_response.status_code)
print('script length:', len(script_response.text))
print('has /analyze/stream:', '/analyze/stream' in script_response.text)
print('has old No Atomization Applied:', 'No Atomization Applied' in script_response.text)
print('has Great Wall sample:', 'Great Wall of China' in script_response.text)
print()
print(script_response.text[:500])


In [ ]:
# Cell 12: inspect server log if needed

server_log_file.flush()

with open(server_log_path, 'r', encoding='utf-8', errors='replace') as log_file:
    print(log_file.read()[-5000:])

print('server still running:', server_process.poll() is None)


In [ ]:
# Cell 13: stop app when finished

from pyngrok import ngrok

if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=10)

if 'server_log_file' in globals() and not server_log_file.closed:
    server_log_file.close()

ngrok.kill()

print('App server and ngrok tunnel stopped.')
